# 🔥 Deep Image Prior: Thermal Image Restoration (Colab GPU Pipeline)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/likhith-naik01/DIP_MP/blob/main/ml/notebooks/colab_run.ipynb)

Restoring degraded thermal images (denoising & inpainting) using unsupervised Deep Image Prior (DIP) with a 5-level U-Net generator.

**All ML computations run 100% in Google Colab on GPU.**

### Cell 1: Check GPU Acceleration

In [ ]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
    device = "cuda"
else:
    print("Running on CPU. To switch to GPU, select Runtime -> Change runtime type -> T4 GPU.")
    device = "cpu"

### Cell 2: Clone Repository

In [ ]:
import os
from pathlib import Path

# Clone or pull latest repository
if not Path("DIP_MP").exists():
    !git clone https://github.com/likhith-naik01/DIP_MP.git
    %cd DIP_MP
else:
    %cd DIP_MP
    !git pull

print(f"Current working directory: {Path.cwd()}")

### Cell 3: Install Package

In [ ]:
# Install dip-core package in editable mode with all dependencies from pyproject.toml
!pip install -q -e ml/

# Verify package import
import dip_core
print("dip_core imported successfully!")

### Cell 4: Mount Google Drive and Setup Output Folder

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

# Setup output directory
output_dir = Path("/content/drive/MyDrive/DIP_outputs")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Results and benchmarks will be saved to: {output_dir}")

### Cell 5: Select / Upload Sample Thermal Image

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
from PIL import Image

print("Option A: Read from repository sample images.")
print("Option B: Upload your own thermal image below (Cancel upload to use sample).\n")

sample_folder = Path("storage/sample_images")
uploaded = files.upload()

if uploaded:
    sample_filename = list(uploaded.keys())[0]
    input_image_path = Path(sample_filename)
    print(f"Using uploaded image: {input_image_path}")
else:
    # Use repo sample
    available = list(sample_folder.glob("*.png"))
    input_image_path = available[0] if available else Path("storage/sample_images/thermal_circuit_board.png")
    print(f"Using repo sample image: {input_image_path}")

### Cell 6: Run U-Net DIP and Conv Baseline, Visualize Restorations and Loss Curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from dip_core.config import DIPConfig
from dip_core.pipeline.preprocess import load_image, simulate_damage
from dip_core.pipeline.trainer import train_dip
from dip_core.pipeline.postprocess import apply_thermal_colormap, save_image
from dip_core.metrics import calculate_psnr, calculate_ssim

# Load image and simulate degradation (noise + missing holes)
clean_img = load_image(str(input_image_path), grayscale=True)
degraded_img, mask = simulate_damage(
    clean_img,
    noise_std=0.10,
    pixel_dropout_rate=0.03,
    add_holes=True,
    hole_type="rect",
)

# 1. Train U-Net DIP (Ours)
print("\n--- Training U-Net DIP Generator ---")
unet_cfg = DIPConfig(
    iterations=1200,
    learning_rate=0.01,
    model_type="unet",
    device=device,
    early_stopping=False,
)
unet_res = train_dip(
    image_input=degraded_img,
    mask_input=mask,
    reference_input=clean_img,
    config=unet_cfg,
    task="inpaint",
)

# 2. Train Conv DIP Baseline
print("\n--- Training Conv DIP Baseline ---")
conv_cfg = DIPConfig(
    iterations=1200,
    learning_rate=0.01,
    model_type="conv_baseline",
    device=device,
    early_stopping=False,
)
conv_res = train_dip(
    image_input=degraded_img,
    mask_input=mask,
    reference_input=clean_img,
    config=conv_cfg,
    task="inpaint",
)

# Visual comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(apply_thermal_colormap(clean_img, "ironbow"))
axes[0].set_title("Original Clean Thermal", fontsize=12, fontweight="bold")
axes[0].axis("off")

p_deg = calculate_psnr(degraded_img, clean_img)
axes[1].imshow(apply_thermal_colormap(degraded_img, "ironbow"))
axes[1].set_title(f"Degraded Input ({p_deg:.2f} dB)", fontsize=12, fontweight="bold")
axes[1].axis("off")

p_conv = conv_res["metrics"]["psnr"]
axes[2].imshow(apply_thermal_colormap(conv_res["restored_image"], "ironbow"))
axes[2].set_title(f"Conv DIP Baseline ({p_conv:.2f} dB)", fontsize=12, fontweight="bold")
axes[2].axis("off")

p_unet = unet_res["metrics"]["psnr"]
axes[3].imshow(apply_thermal_colormap(unet_res["restored_image"], "ironbow"))
axes[3].set_title(f"U-Net DIP Ours ({p_unet:.2f} dB)", fontsize=12, fontweight="bold")
axes[3].axis("off")

plt.tight_layout()
comp_out_file = output_dir / "single_restoration_comparison.png"
plt.savefig(str(comp_out_file), dpi=150, bbox_inches="tight")
plt.show()

# Loss Convergence Curves
plt.figure(figsize=(9, 4.5))
plt.plot(unet_res["loss_history"], label="U-Net DIP (Ours)", color="#f97316", linewidth=2)
plt.plot(conv_res["loss_history"], label="Conv DIP (Baseline)", color="#3b82f6", linewidth=1.8, linestyle="--")
plt.yscale("log")
plt.xlabel("Iteration", fontsize=11)
plt.ylabel("Loss (Log Scale)", fontsize=11)
plt.title("DIP Loss Convergence Comparison", fontsize=13, fontweight="bold")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(fontsize=11)
loss_out_file = output_dir / "loss_convergence_curve.png"
plt.savefig(str(loss_out_file), dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved plots to Drive: {output_dir}")

### Cell 7: Run Benchmark and Save results.csv & Comparison PNGs to Google Drive

In [ ]:
# Run full benchmark across thermal images
!python ml/scripts/benchmark.py --samples storage/sample_images --output "/content/drive/MyDrive/DIP_outputs" --iters 1000 --device cuda

# List files saved in Google Drive
print("\n--- Files saved in Google Drive ---")
for f in output_dir.glob("*"):
    print(f"  • {f.name}")

### Optional Cell 8: Run FastAPI Backend with Tunnel in Colab (for Web Frontend Demo)
Run this cell if you want to connect your local `frontend/index.html` to this Colab GPU session via a public HTTPS tunnel.

In [ ]:
!pip install -q fastapi uvicorn python-multipart
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess
import time
import re

print("Starting FastAPI server in background...")
backend_proc = subprocess.Popen([
    "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)

print("Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

tunnel_url = None
start_t = time.time()
while time.time() - start_t < 25:
    line = tunnel_proc.stderr.readline().decode('utf-8')
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

print("\n===================================================================")
if tunnel_url:
    print(f"🚀 COLAB GPU BACKEND IS READY!")
    print(f"👉 {tunnel_url}")
    print("\nPaste this URL into the Server URL box at the top of frontend/index.html!")
else:
    print("Tunnel started. Check tunnel logs for URL.")
print("===================================================================")